### Balanced Label Creation
> - cens_dfs == 1 -> event occured (recurrence, death, etc.)
> - cens_dfs == 0 -> censored (no event observed, maybe lost to follow-up or study ended)

We want to focus only on patients with an event occuring (cens_dfs == 1) and compare their DFS time.

In [37]:
!pip install ete3 torch lifelines

In [38]:
import os
import re
import random
from collections import defaultdict
from lifelines.utils import concordance_index
import numpy as np
import pandas as pd
from ete3 import Tree
import math
import random
from typing import List, Dict, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report

In [39]:
import os
!git clone https://github.com/szinja/cell-translation.git
%cd cell-translation
%ls

Cloning into 'cell-translation'...
remote: Enumerating objects: 992, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 992 (delta 42), reused 49 (delta 14), pack-reused 903 (from 2)
Receiving objects: 100% (992/992), 74.09 MiB | 14.79 MiB/s, done.
Resolving deltas: 100% (337/337), done.
/content/cell-translation/cell-translation
clinical_data/               merged_patient_data_with_subclonal_scores.csv
cox.ipynb                    multi_branch_deepsurv.ipynb
data/                        patient_embeddings_with_labels.csv
data_analysis.ipynb          patient_subclonal_expansion_scores.csv
deepsurv.ipynb               random_forest.ipynb
deepsurv_model.pt            random_forest_predictions.ipynb
embedding_scaler.joblib      regression_models.ipynb
experiments.ipynb            tree_and_clinical_data_lstms.ipynb
final_model.ipynb            trees.txt
indices_cox_model.ipynb      tree_tumourid.csv
logistical_regression.ipynb

Our aim is to predict overall survival (OS).

From TracerX description:
* Duration indicator: `os_time`
* Event indicatior: `cens_os` indicates whether the patient was censored (`1` means censored, and `0` means event occurred/death).

Since `cens_os` flags censoring due to death with 0, we will invert it `event_os = 1 - cens_os` to create the event indicator for overall survival analysis:
* `event_os = 1` → death (event of interest)

* `event_os = 0` → censored

In [40]:
# Datasets
tracerx_df = pd.read_csv("data/20221109_TRACERx421_all_patient_df_Converted.csv")
tracerx_df.drop(columns=['Unnamed: 0'], inplace=True)
tracerx_df.columns = tracerx_df.columns.str.strip()

# Rename 'patient_id' to 'cruk_id'
if 'patient_id' in tracerx_df.columns:
    tracerx_df = tracerx_df.rename(columns={'patient_id': 'cruk_id'})

# Format 'cruk_id'
def format_cruk_id(x):
    try:
        return f"CRUK{int(x):04d}"
    except (ValueError, TypeError):
        return x

tracerx_df['cruk_id'] = tracerx_df['cruk_id'].apply(format_cruk_id)

print(f"Shape of TracerX Dataset: {tracerx_df.shape}")
print(tracerx_df.head())

Shape of TracerX Dataset: (421, 45)
    cruk_id  age  sex  ethnicity  cigs_perday  years_smoking  packyears  \
0  CRUK0032   68    0          8         20.0             35     35.000   
1  CRUK0107   81    1          7         44.5             49    109.025   
2  CRUK0109   60    1          7         20.0             38     38.000   
3  CRUK0087   65    1          7         10.0             35     17.500   
4  CRUK0043   85    1          7         10.0             25     12.500   

   smoking_status_merged  is.family.lung  ECOG_PS  ...  os_time  cens_dfs  \
0                      0               1      0.0  ...     1849         0   
1                      0               0      0.0  ...     1362         1   
2                      2               0      0.0  ...     2224         1   
3                      0               0      1.0  ...     2365         1   
4                      0               0      1.0  ...      986         1   

   dfs_time  cens_dfs_any_event  dfs_time_any_even

In [41]:
# From TracerX dataset, we want to create a balanced dataset based on OS time.
tracerx_df.columns = tracerx_df.columns.str.strip()
TIME_THRESHOLD_DAYS = 365*3.5

# Drop rows with missing essential survival data
tracerx_df.dropna(subset=['os_time', 'cens_os'], inplace=True)

# 1 = death (event occurred), 0 = censored (alive)
tracerx_df['event_os'] = 1 - tracerx_df['cens_os']
tracerx_df['os_risk_label'] = pd.NA # creates binary classification labeling

# Define the HIGH-RISK group: Event (death) occurred BEFORE the time threshold
tracerx_df.loc[
    (tracerx_df['os_time'] < TIME_THRESHOLD_DAYS) & (tracerx_df['event_os'] == 1),
    'os_risk_label'
] = 1

# Define the LOW-RISK group: No event, and follow-up time is AFTER the threshold
tracerx_df.loc[
    (tracerx_df['os_time'] >= TIME_THRESHOLD_DAYS) & (tracerx_df['event_os'] == 0),
    'os_risk_label'
] = 0

# Create a dataframe that drops the ambiguous cases
combined_df = tracerx_df.dropna(subset=['os_risk_label']).copy()

# Convert the label to a clean integer type
combined_df['os_risk_label'] = combined_df['os_risk_label'].astype(int)

print(f"Original dataset size: {len(tracerx_df)}")
print(f"Final binary classification dataset size: {len(combined_df)}")
print("\nDistribution of the new 'os_risk_label':")
print(combined_df['os_risk_label'].value_counts())

Original dataset size: 421
Final binary classification dataset size: 80

Distribution of the new 'os_risk_label':
os_risk_label
1    51
0    29
Name: count, dtype: int64


The TracerX dataset goes from 421 to 80 patients. This reduction happens because we excluded the "ambiguous" patients:

- Those who were censored before the 3.5-year mark (we don't know their outcome at the threshold).

- Those who died after the 3.5-year mark (they don't fit the "high-risk" definition of dying before the threshold).






---


### Example Tree and clinical string encodings
>- Newick Tree: ((1,11)8,(14,12)9)root; etc.
>- Clinical data: ("age_65 sex_M ECOG_PS_1 smoking_20py"...) etc.

For simplicity, we consider only the primary T1 tumor for every patient.

In [42]:
newick_dir = "trees.txt"  # patient newick trees
cruk_id_col = "cruk_id"
newick_file_template = "{cruk_id}.newick"  # e.g., CRUK0361.newick
random_seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 60
lr = 1e-3
embed_dim = 32
mem_dim = 64
clinical_feature_cols = None  # e.g. ['age', 'packyears']

In [43]:
# Load mapping
tree_id_map = pd.read_csv("tree_tumourid.csv", index_col=0)
tree_id_map.columns = tree_id_map.columns.str.strip()

# Filter out Tumour2
tree_id_map_filtered = tree_id_map[~tree_id_map['x'].str.contains('Tumour2')].reset_index(drop=True)

# Load Newick trees
with open("trees.txt") as f:
    newick_lines = [line.strip() for line in f if line.strip()]

# Build mapping - filtered IDs only
trees_by_crukid = dict(zip(tree_id_map_filtered['x'], newick_lines))

print(f"Loaded {len(trees_by_crukid)} trees after filtering")

Loaded 392 trees after filtering


In [44]:
removed_ids = sorted(set(tree_id_map['x']) - set(tree_id_map_filtered['x']))
print(f"{len(removed_ids)} IDs removed: {removed_ids}")

9 IDs removed: ['CRUK0030_Tumour2', 'CRUK0223_Tumour2', 'CRUK0372_Tumour2', 'CRUK0555_Tumour2', 'CRUK0586_Tumour2', 'CRUK0620_Tumour2', 'CRUK0704_Tumour2', 'CRUK0721_Tumour2', 'CRUK0881_Tumour2']


### Tree-LSTMs
Parsing a Newick tree string into a Tree-LSTM object.

In [45]:
class TreeNode:
    def __init__(self, name=None):
        self.name = name
        self.children = []
        self.idx = None

def parse_newick_to_treenode(newick_str):
    ete_tree = Tree(newick_str, format=1)

    def build_node(ete_node):
        node = TreeNode(name=ete_node.name if ete_node.name else None)
        node.children = [build_node(child) for child in ete_node.children]
        return node

    return build_node(ete_tree)

In [46]:
def parse_newick_to_nodes_and_children(newick_text: str) -> Tuple[List[str], List[List[int]]]:
    t = Tree(newick_text, format=1)  # using ete3
    node_list = []
    children = []

    # Map node object -> index
    idx_map = {}
    idx = 0

    # Do a traversal and assign indices
    for n in t.traverse("preorder"):
        idx_map[n] = idx
        # use node name if present, else use empty string
        node_list.append(n.name if n.name is not None else "")
        children.append([])
        idx += 1

    # fill children arrays: ete3 n.children gives node objects
    for n in t.traverse("preorder"):
        parent_idx = idx_map[n]
        for ch in n.children:
            children[parent_idx].append(idx_map[ch])

    # Ensure root is index 0 - etree preorder gives root first so ok
    return node_list, children

# Node feature encoder
#   label embedding: map string token -> integer id -> nn.Embedding
class NodeLabelVocab:
    def __init__(self):
        self.token2idx = {"<PAD>": 0}
        self.idx2token = {0: "<PAD>"}
        self.next_idx = 1

    def add(self, token: str):
        if token not in self.token2idx:
            self.token2idx[token] = self.next_idx
            self.idx2token[self.next_idx] = token
            self.next_idx += 1
        return self.token2idx[token]

    def token_to_idx(self, token: str) -> int:
        return self.token2idx.get(token, 0)  # fallback to <PAD> for unknown

    def idx_to_token(self, idx: int) -> str:
        return self.idx2token.get(idx, "<PAD>")

    def __len__(self):
        return len(self.token2idx)


# Build token vocab by scanning Newick files listed in the tracerx dataframe
def build_label_vocab_from_df(df: pd.DataFrame, newick_dir: str, cruk_col: str = "cruk_id", file_template: str = "{cruk_id}.newick"):
    vocab = NodeLabelVocab()
    for cruk in df[cruk_col].dropna().unique():
        fpath = os.path.join(newick_dir, file_template.format(cruk_id=cruk))
        if not os.path.exists(fpath):
            continue
        try:
            with open(fpath, "r") as fh:
                newick = fh.read().strip()
            node_names, _ = parse_newick_to_nodes_and_children(newick)
            for nm in node_names:
                vocab.add(nm)
        except Exception as e:
            # skip unreadable
            print("Warning: couldn't read", fpath, ":", e)
            continue
    return vocab

# Build vocab
vocab = build_label_vocab_from_df(
    tracerx_df,
    newick_dir=newick_dir,
    cruk_col=cruk_id_col,
    file_template=newick_file_template
)


print("Built node label vocab size:", len(vocab))

# Dataset that returns (nodes_tensor, children_list, clinical_tensor, time, event)
# Node features: either numeric (if token parseable) OR token embedding indices (int list).
# We'll produce:
#   nodes_tensor: torch.FloatTensor (n_nodes, node_feat_dim)
# For embedding-mode, node_feat_dim will be node_embedding_dim (set below).
class TreeClinicalDataset(torch.utils.data.Dataset):
    def __init__(self, samples, vocab, clinical_cols=None):
        """
        samples: list of dicts with keys:
            - node_names: list of node labels
            - children: list of child indices
            - clinical: dict of clinical features
            - time: float
            - event: float
        """
        self.samples = samples
        self.vocab = vocab
        self.clinical_cols = clinical_cols or []

        # Compute mean/std for clinical normalization
        if self.clinical_cols:
            clin_matrix = np.array([[s["clinical"].get(c, 0.0) for c in self.clinical_cols] for s in samples], dtype=np.float32)
            self.clin_mean = clin_matrix.mean(axis=0)
            self.clin_std = clin_matrix.std(axis=0) + 1e-8
        else:
            self.clin_mean = np.zeros(0)
            self.clin_std = np.ones(0)

    def __len__(self):
        return len(self.samples)

    def _node_list_to_tensor(self, node_names):
        idxs = [self.vocab.token2idx.get(nm, 0) for nm in node_names]
        return torch.tensor(idxs, dtype=torch.long)

    def __getitem__(self, idx):
        s = self.samples[idx]
        nodes_tensor = self._node_list_to_tensor(s["node_names"])
        children = s["children"]

        if self.clinical_cols:
            arr = np.array([s["clinical"].get(c, 0.0) for c in self.clinical_cols], dtype=np.float32)
            arr = (arr - self.clin_mean) / self.clin_std
            clin_tensor = torch.tensor(arr, dtype=torch.float32)
        else:
            clin_tensor = torch.tensor([], dtype=torch.float32)

        return nodes_tensor, children, clin_tensor, torch.tensor(s["time"], dtype=torch.float32), torch.tensor(s["event"], dtype=torch.float32)

# Collate - return lists for trees (we handle per-sample tree processing in model loop)
def collate_trees_clinical(batch):
    roots, clin_list, times_list, events_list = zip(*batch)

    # Stack clinical features, time, and event
    if clin_list[0].numel() > 0:
        clin_batch = torch.cat(clin_list, dim=0)
    else:
        clin_batch = torch.tensor([], dtype=torch.float32)

    times_batch = torch.stack(times_list)
    events_batch = torch.stack(events_list)

    return roots, clin_batch, times_batch, events_batch

# TreeSurvivalModel (adapted to accept clinical vector appended to root embedding)
# Uses a learnable embedding for node labels in label mode, or uses numeric node features directly.
class TreeSurvivalModelWithClinical(nn.Module):
    def __init__(self, vocab_size, node_emb_dim, tree_hid, mlp_hidden, clinical_dim, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, node_emb_dim)
        self.cell = ChildSumTreeLSTMCell(node_emb_dim, tree_hid)
        # Add Layer Normalization for the TreeLSTM output
        self.ln = nn.LayerNorm(tree_hid)
        self.mlp = nn.Sequential(
            nn.Linear(tree_hid + clinical_dim, mlp_hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1)
        )

    def forward(self, tree, clinical_tensor=None):
        device = self.emb.weight.device
        def recurse(node):
            # Print the value and type of node.idx before creating the tensor
            # print(f"node.idx value: {getattr(node, 'idx', 'N/A')}, type: {type(getattr(node, 'idx', None))}")
            idx = torch.tensor([node.idx if node.name is not None else 0],
                               dtype=torch.long, device=device)
            # Print the index being used for embedding lookup
            # print(f"Embedding lookup index: {idx.item()}") # Commented out to reduce output clutter
            embed = self.emb(idx)  # shape [1, embed_dim], keeps requires_grad=True automatically
            if torch.isnan(embed).any():
                print(f"NaN detected in 'embed' for node {getattr(node, 'name', 'N/A')}")
            # Assert only if gradients is enabled
            if torch.is_grad_enabled():
                assert embed.requires_grad, "Embedding tensor within recurse does not require gradients!"


            child_c, child_h = [], []
            for child in getattr(node, "children", []):
                c, h = recurse(child)
                child_c.append(c)
                child_h.append(h)

            # Pass through TreeLSTM cell
            c, h = self.cell(embed, child_c, child_h)

            if torch.isnan(c).any():
              print(f"NaN detected in cell state 'c' for node {getattr(node, 'name', 'N/A')}")
            if torch.isnan(h).any():
              print(f"NaN detected in hidden state 'h' for node {getattr(node, 'name', 'N/A')}")

            # Add more aggressive clipping to intermediate c and h
            c = torch.clamp(c, min=-1e2, max=1e2)
            h = torch.clamp(h, min=-1e2, max=1e2)


            # Assert only if gradients is enabled
            if torch.is_grad_enabled():
                assert c.requires_grad, "Cell state 'c' within recurse does not require gradients!"
                assert h.requires_grad, "Hidden state 'h' within recurse does not require gradients!"

            return c, h

        c_root, h_root = recurse(tree)

        # Apply Layer Normalization to h_root
        h_root = self.ln(h_root)
        # Add more aggressive clipping to h_root after Layer Norm
        h_root = torch.clamp(h_root, min=-1e2, max=1e2)


        # Handle clinical tensor
        if clinical_tensor is None or clinical_tensor.numel() == 0:
            clin = torch.zeros((1, 0), device=h_root.device)
        else:
            clin = clinical_tensor.to(h_root.device)
            if clin.dim() == 1:
                clin = clin.unsqueeze(0)

        # Debugging assertions (conditional)
        if torch.is_grad_enabled():
            assert h_root.requires_grad, "h_root does not require gradients!"


        x = torch.cat([h_root, clin], dim=1)

        # Debugging assertion (conditional)
        if torch.is_grad_enabled():
            assert x.requires_grad, "Concatenated tensor 'x' does not require gradients!"

        # Apply MLP layers
        mlp_output = self.mlp(x)

        # Add more aggressive clipping to the final MLP output (risk score)
        risk = torch.clamp(mlp_output.squeeze(-1), min=-1e2, max=1e2)

        return risk

def cox_ph_loss(risk: torch.Tensor, times: torch.Tensor, events: torch.Tensor):
    """
    Cox partial likelihood loss (negative average log-likelihood).
    Keeps computation graph intact even if no events are present in the batch.
    """
    # ensure 1D
    risk = risk.view(-1)
    times = times.view(-1)
    events = events.view(-1).float()

    # sort by descending time
    order = torch.argsort(times, descending=True)
    risk_sorted = risk[order]
    times_sorted = times[order]
    events_sorted = events[order]

    # denom: log cumulative sum exp(risk) in a stable way
    try:
        denom = torch.logcumsumexp(risk_sorted, dim=0)
    except Exception:
        exp_r = torch.exp(risk_sorted - risk_sorted.max())
        cums = torch.cumsum(exp_r, dim=0)
        denom = torch.log(cums) + risk_sorted.max()

    # Initialize loss as zero but attached to the graph (safe if there are no events)
    loss = risk.sum() * 0.0  # <-- important: has grad_fn thanks to `risk`

    n = risk_sorted.size(0)
    i = 0
    while i < n:
        t_i = times_sorted[i]
        j = i
        # use .item() for Python-level control (safe)
        while j < n and times_sorted[j].item() == t_i.item():
            j += 1

        # number of events at this time
        d = events_sorted[i:j].sum()
        if d.item() > 0:   # use .item() to check value in python if-block
            # sum of risks for individuals who experienced event at this time
            mask = events_sorted[i:j] == 1
            r_events = risk_sorted[i:j][mask].sum()
            denom_block = denom[j - 1]  # log-sum-exp up to j-1
            # accumulate negative log-likelihood piece
            loss = loss - (r_events - d * denom_block)

        i = j

    n_events = events.sum()
    if n_events.item() > 0:
        loss = loss / n_events

    return loss

# Concordance index (naive O(N^2) for clarity)
def concordance_index(pred: np.ndarray, times: np.ndarray, events: np.ndarray):
    # Returns C-index (higher better)
    n = len(pred)
    assert len(times) == n and len(events) == n
    permissible = 0
    concordant = 0
    tied = 0
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if times[i] < times[j] and events[i] == 1:
                permissible += 1
                if pred[i] > pred[j]:
                    concordant += 1
                elif pred[i] == pred[j]:
                    tied += 1
    if permissible == 0:
        return 0.5
    return (concordant + 0.5 * tied) / permissible


# Random dataset creation
def make_random_tree(max_nodes=8, feat_dim=4, p_branch=0.5):
    # Create a random tree: root index 0
    # We'll create nodes incrementally, randomly assign each new node to be child of an existing node
    n_nodes = random.randint(1, max_nodes)
    nodes = []
    children = [[] for _ in range(n_nodes)]
    for i in range(n_nodes):
        # create random node feature
        feat = np.random.randn(feat_dim).astype(np.float32)
        nodes.append(feat)
        if i > 0:
            # attach as child to a random previous node
            parent = random.randint(0, i - 1)
            children[parent].append(i)
    nodes_t = torch.tensor(np.stack(nodes), dtype=torch.float32)
    return nodes_t, children

Built node label vocab size: 1


In [47]:
# Helper function to convert TreeNode to a list of indices
def collect_names(node, names):
    if node.name:
        names.add(node.name)
    for c in node.children:
        collect_names(c, names)

def assign_indices(node, vocab):
    # Get index from vocab, default to 0 if not found
    idx = vocab.token_to_idx(node.name) if node.name else 0
    # Add a check to ensure the index is within the valid range
    if idx >= len(vocab):
        print(f"Warning: Assigned index {idx} for node '{node.name}' is out of vocabulary range ({len(vocab)}). Setting to padding index (0).")
        node.idx = 0
    else:
        node.idx = idx
    for c in getattr(node, "children", []):
        assign_indices(c, vocab)

In [48]:
# 1. Initialize empty vocab
vocab = NodeLabelVocab()

# 2. Collect all node names from trees
names = set()
for newick_str in trees_by_crukid.values():
    root = parse_newick_to_treenode(newick_str)
    collect_names(root, names)

# 3. Add names to NodeLabelVocab
for name in sorted(names):
    vocab.add(name)

print(f"Built NodeLabelVocab size: {len(vocab)}")

# 4. Assign indices to all tree nodes
for newick_str in trees_by_crukid.values():
    root = parse_newick_to_treenode(newick_str)
    assign_indices(root, vocab)

Built NodeLabelVocab size: 58


In [49]:
# TreeLSTM dataset class
class ChildSumTreeLSTMCell(nn.Module):
    def __init__(self, in_dim, mem_dim):
        super().__init__()
        self.in_dim = in_dim
        self.mem_dim = mem_dim
        # Input + hidden to gates
        self.W_iou = nn.Linear(in_dim, 3 * mem_dim)
        self.U_iou = nn.Linear(mem_dim, 3 * mem_dim, bias=False)
        self.U_f = nn.Linear(mem_dim, mem_dim, bias=False)
        self.b_f = nn.Parameter(torch.zeros(mem_dim))

    def forward(self, inputs, child_c, child_h):
        # Assert only if gradients are enabled
        if torch.is_grad_enabled():
            assert inputs.requires_grad, "Input tensor to ChildSumTreeLSTMCell does not require gradients!"
            for i, hc in enumerate(zip(child_h, child_c)):
                assert hc[0].requires_grad, f"Child hidden state {i} does not require gradients!"
                assert hc[1].requires_grad, f"Child cell state {i} does not require gradients!"

        if len(child_h) == 0:
            h_sum = torch.zeros(1, self.mem_dim, device=inputs.device)
        else:
            h_sum = torch.sum(torch.stack(child_h, dim=0), dim=0)

        # Ensure h_sum has gradients if children have gradients (assert only if gradients are enabled)
        if torch.is_grad_enabled() and len(child_h) > 0:
             assert h_sum.requires_grad == any(ch.requires_grad for ch in child_h), "h_sum gradient tracking issue!"


        iou = self.W_iou(inputs) + self.U_iou(h_sum)
        i, o, u = torch.chunk(iou, 3, dim=1)
        i = torch.sigmoid(i)
        o = torch.sigmoid(o)
        u = torch.tanh(u)

        if len(child_h) == 0:
            c = i * u
        else:
            f_list = [torch.sigmoid(self.U_f(h) + self.b_f) for h in child_h]
            fc = torch.sum(torch.stack([f * c for f, c in zip(f_list, child_c)], dim=0), dim=0)
            c = i * u + fc

        h = o * torch.tanh(c) # Calculate h here

        # Assert only if gradients are enabled
        if torch.is_grad_enabled():
            assert c.requires_grad, "Cell state 'c' does not require gradients!"
            assert h.requires_grad, "Hidden state 'h' does not require gradients!"


        return c, h

In [50]:
class TreeLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, mem_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.cell = ChildSumTreeLSTMCell(embed_dim, mem_dim)

    def forward(self, tree):
        device = self.emb.weight.device
        # Node embedding
        idx = torch.tensor([tree.idx if tree.name is not None else 0],
                           dtype=torch.long, device=device)
        emb = self.emb(idx)  # shape [1, embed_dim], keeps requires_grad=True automatically
        # Assert only if gradients are enabled
        if torch.is_grad_enabled():
             assert emb.requires_grad, "Embedding tensor does not require gradients!"

        # Recursively get child states
        child_c, child_h = [], []
        for child in getattr(tree, "children", []):
            c, h = self.forward(child)
            child_c.append(c)
            child_h.append(h)

        # Pass through TreeLSTM cell
        c, h = self.cell(emb, child_c, child_h)
        return c, h

In [51]:
# Display first few trees
for cruk_id, newick in list(trees_by_crukid.items())[:5]:
    print(f"{cruk_id}: {newick}")


CRUK0005: (((((((8:24)18:7)12:3,10:10)13:13)9:159,((((17:4)21:89)5:87)19:2)20:178)3:7)4:82,1:203)2;
CRUK0057: ((4:23)2:999,((6:1)5:24)3:1007)1;
CRUK0039: ((4:34)2:958,(5:24)3:971)1;
CRUK0196: (((7:2,8:2)6:12)2:631,(5:4)4:636)1;
CRUK0023: ((((12:11)4:13)13:20)3:195,((8:1e-06)9:2,(10:1e-06)11:3)7:209,2:189)1;


In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print vocab size and embedding layer size before moving to device
print(f"Vocabulary size: {len(vocab)}")
model = TreeLSTM(vocab_size=len(vocab), embed_dim=16, mem_dim=32)
print(f"Embedding layer weight size before to(device): {model.emb.weight.size()}")

model = model.to(device)
# model.eval() # Removed model.eval() here

patient_embeddings = {}
with torch.no_grad():
    for cruk_id, newick_str in trees_by_crukid.items():
        root = parse_newick_to_treenode(newick_str)
        assign_indices(root, vocab)
        c, h = model(root)
        patient_embeddings[cruk_id] = h.squeeze(0).cpu()

# Convert to DataFrame
df_embeddings = pd.DataFrame.from_dict(
    {pid: emb.numpy() for pid, emb in patient_embeddings.items()},
    orient="index"
).reset_index().rename(columns={"index": "cruk_id"})

Vocabulary size: 58
Embedding layer weight size before to(device): torch.Size([58, 16])


In [53]:
# Merge embeddings with labels
final_df = df_embeddings.merge(
    combined_df[['cruk_id', 'os_risk_label']],
    on="cruk_id",
    how="inner"
)

print("Final dataset shape:", final_df.shape)
print(final_df.head())

# Save for downstream modeling
final_df.to_csv("patient_embeddings_with_labels.csv", index=False)

Final dataset shape: (43, 34)
    cruk_id         0         1         2         3         4         5  \
0  CRUK0276 -0.056752 -0.131186  0.225776  0.121452 -0.011450  0.184197   
1  CRUK0077 -0.017741 -0.116164  0.286932  0.036594  0.091892  0.235183   
2  CRUK0020  0.034416 -0.142450  0.335909  0.097462  0.077083  0.154303   
3  CRUK0036  0.013150  0.075048  0.296074  0.050584  0.067896  0.143237   
4  CRUK0178 -0.058206 -0.030134  0.234743  0.177812 -0.041894  0.163795   

          6         7         8  ...        23        24        25        26  \
0 -0.094232  0.095933 -0.099956  ... -0.043179 -0.108107 -0.144100  0.119084   
1 -0.188789 -0.155503 -0.241890  ... -0.031768 -0.010969 -0.289285  0.128556   
2 -0.148775  0.045030 -0.168211  ... -0.039064 -0.071279 -0.326664  0.031864   
3 -0.054087 -0.125757 -0.118475  ... -0.010528  0.120768 -0.253210  0.171695   
4 -0.145540  0.158595 -0.183452  ... -0.047620 -0.084490 -0.202354  0.148088   

         27        28        29       

In [54]:
import pandas as pd

def read_trees_lines(trees_txt_path: str) -> List[str]:
    """Read trees.txt where each line is a Newick string."""
    with open(trees_txt_path, "r") as fh:
        lines = [ln.strip() for ln in fh if ln.strip()]
    return lines

def build_label_vocab_from_mapping(df: pd.DataFrame, trees: List[str], id_col: str = "cruk_id", line_col: str = "line_index"):
    """
    Build vocab by looking up each patient's tree from trees.txt using line mapping.
    - df must have columns [id_col, line_col]
    """
    vocab = NodeLabelVocab()
    for _, row in df.iterrows():
        line_idx = int(row[line_col])
        if line_idx < 0 or line_idx >= len(trees):
            continue
        newick = trees[line_idx]
        try:
            node_names, _ = parse_newick_to_nodes_and_children(newick)
            for nm in node_names:
                vocab.add(nm)
        except Exception as e:
            print(f"Warning: could not parse line {line_idx} for {row[id_col]}: {e}")
            continue
    return vocab


class TreeClinicalDatasetLines(torch.utils.data.Dataset):
    def __init__(self, df, trees_lines, cruk_col="cruk_id", line_col="line_index",
                 time_col="os_time", event_col="event_os",
                 clinical_cols=None, vocab=None):
        """
        df: DataFrame with patient info and mapping to tree lines
        trees_lines: list of Newick strings, indexed by line
        clinical_cols: optional list of clinical feature names
        vocab: NodeLabelVocab object
        """
        self.vocab = vocab
        self.trees_lines = trees_lines
        self.clinical_cols = clinical_cols if clinical_cols else []
        self.samples = []

        # Prepare mean/std for normalization if clinical features are used
        if self.clinical_cols:
            self.clin_mean = df[self.clinical_cols].mean()
            self.clin_std  = df[self.clinical_cols].std().replace(0, 1)  # avoid div0
        else:
            self.clin_mean = self.clin_std = None

        # Build dataset samples
        for _, row in df.iterrows():
            line_idx = int(row[line_col])
            if line_idx < 0 or line_idx >= len(trees_lines):
                continue
            newick_str = trees_lines[line_idx]
            root = parse_newick_to_treenode(newick_str)
            assign_indices(root, self.vocab)

            clinical_data = {c: row[c] for c in self.clinical_cols} if self.clinical_cols else {}

            self.samples.append({
                "cruk_id": row[cruk_col],
                "root": root,
                "clinical": clinical_data,
                "time": row[time_col],
                "event": row[event_col]
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        root = s["root"]

        # Clinical tensor
        if self.clinical_cols:
            arr = np.array([s["clinical"].get(c, 0.0) for c in self.clinical_cols], dtype=np.float32)
            arr = (arr - self.clin_mean.values) / self.clin_std.values
            clin_tensor = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)  # shape [1, num_features]
        else:
            clin_tensor = torch.tensor([], dtype=torch.float32)

        return root, clin_tensor, torch.tensor(s["time"], dtype=torch.float32), torch.tensor(s["event"], dtype=torch.float32)

In [55]:
# Columns we must not use as clinical features
exclude_cols = {
    "cruk_id", "tumour_id_muttable_cruk", "tumour_id_per_patient",
    "dfs_time", "cens_dfs", "dfs_time_any_event",
    "cens_dfs_any_event", "os_time", "cens_lung_event",
    "lung_event_time", "Relapse_cat", "Relapse_cat_new",
    "tx100", "line_index", "cens_os", "event_os"
}

# Auto-detect numeric clinical columns
clinical_feature_cols = [
    col for col in tracerx_df.select_dtypes(include=[np.number]).columns
    if col not in exclude_cols
]

print("Detected clinical features:", clinical_feature_cols)
print("Number of clinical features:", len(clinical_feature_cols))

Detected clinical features: ['age', 'sex', 'ethnicity', 'cigs_perday', 'years_smoking', 'packyears', 'smoking_status_merged', 'is.family.lung', 'ECOG_PS', 'pathologyTNM', 'pT_stage_per_patient', 'pN_stage_per_patient', 'LVI_per_patient', 'size_pathology_per_patient', 'Surgery_type', 'histology_lesion1', 'lesion1_sampled', 'histology_lesion2', 'lesion2_sampled', 'histology_multi_full', 'histology_multi_full_genomically.confirmed', 'LUAD_pred_subtype', 'adjuvant_treatment_YN', 'adjuvant_treatment_given', 'num_cycle_na.added', 'CHMPlatDgName_cleaned', 'CHMOthDgName_cleaned', 'AdjRadStartTime_manual', 'AdjRadEndTime_manual', 'Recurrence_time_use', 'newPrim_time_use', 'first_dfs_any_event_rec.or.new.primary', 'first_event_during_followup']
Number of clinical features: 33


In [56]:
print("Checking for NaNs in clinical columns before imputation...")
print(tracerx_df[clinical_feature_cols].isnull().sum())

# Calculate the mean for each clinical column
imputation_values = tracerx_df[clinical_feature_cols].mean()

# Fill the NaNs in the DataFrame with the calculated means
tracerx_df[clinical_feature_cols] = tracerx_df[clinical_feature_cols].fillna(imputation_values)

print("\nChecking for NaNs after imputation...")
print(tracerx_df[clinical_feature_cols].isnull().sum())

Checking for NaNs in clinical columns before imputation...
age                                             0
sex                                             0
ethnicity                                       0
cigs_perday                                     0
years_smoking                                   0
packyears                                       0
smoking_status_merged                           0
is.family.lung                                  0
ECOG_PS                                         0
pathologyTNM                                    0
pT_stage_per_patient                            0
pN_stage_per_patient                            0
LVI_per_patient                                 0
size_pathology_per_patient                      0
Surgery_type                                    0
histology_lesion1                               0
lesion1_sampled                                 0
histology_lesion2                               0
lesion2_sampled                          

In [57]:
# 1. Load tree lines
trees = read_trees_lines("trees.txt")

# 2. Load mapping + fix columns
tree_map = pd.read_csv("tree_tumourid.csv")
tree_map = tree_map.rename(columns={"x": "cruk_id", "Unnamed: 0": "line_index"})
tree_map["line_index"] -= 1

# 3. Merge mapping into clinical df
print("Tracerx rows before merge:", len(tracerx_df))

tracerx_df = tracerx_df.merge(tree_map, on="cruk_id", how="inner")

tracerx_df = tracerx_df[tracerx_df["line_index"] < len(trees)]


print("Tracerx rows after merge:", len(tracerx_df))

# 4. Build vocab
vocab = build_label_vocab_from_mapping(tracerx_df, trees, id_col="cruk_id", line_col="line_index")
print("Built node label vocab size:", len(vocab))
print("Trees loaded:", len(trees))
print("Line index min/max:", tracerx_df["line_index"].min(), tracerx_df["line_index"].max())
print("Any bad indices?", (tracerx_df["line_index"] >= len(trees)).sum())

# 5. Build dataset
dataset = TreeClinicalDatasetLines(
    tracerx_df, trees,
    cruk_col="cruk_id", line_col="line_index",
    time_col="os_time", event_col="event_os",
    clinical_cols=clinical_feature_cols,
    vocab=vocab
)

# Check if CRUK IDs overlap
print("Unique CRUK IDs in tracerx_df:", tracerx_df["cruk_id"].nunique())
print("Unique CRUK IDs in tree_map:", tree_map["cruk_id"].nunique())
print("Overlap:", tracerx_df["cruk_id"].isin(tree_map["cruk_id"]).sum())

print("Dataset size:", len(dataset))


Tracerx rows before merge: 421
Tracerx rows after merge: 212
Built node label vocab size: 58
Trees loaded: 400
Line index min/max: 0 398
Any bad indices? 0
Unique CRUK IDs in tracerx_df: 212
Unique CRUK IDs in tree_map: 401
Overlap: 212
Dataset size: 212


In [58]:
train_idx, val_idx = train_test_split(
    list(range(len(dataset))),
    test_size=0.2,
    random_state=random_seed
)

In [59]:
# -----------------------
# 1. Node mode + clinical columns
# -----------------------
node_mode = "label" if len(vocab) > 1 else "numeric"

clinical_cols = clinical_feature_cols if clinical_feature_cols else []

# -----------------------
# 2. Dataset (line-based)
# -----------------------
dataset = TreeClinicalDatasetLines(
    tracerx_df, trees,
    cruk_col="cruk_id", line_col="line_index",
    time_col="os_time", event_col="event_os",
    clinical_cols=clinical_cols,
    vocab=vocab
)

print(f"Dataset size: {len(dataset)}")

Dataset size: 212


In [60]:
# Save the final_df DataFrame (patient embeddings with labels) to a CSV file
output_csv_path = "final_patient_ds.csv"
final_df.to_csv(output_csv_path, index=False)

print(f"Saved final dataset to {output_csv_path}")

Saved final dataset to final_patient_ds.csv


In [61]:
%ls

clinical_data/               merged_patient_data_with_subclonal_scores.csv
cox.ipynb                    multi_branch_deepsurv.ipynb
data/                        patient_embeddings_with_labels.csv
data_analysis.ipynb          patient_subclonal_expansion_scores.csv
deepsurv.ipynb               random_forest.ipynb
deepsurv_model.pt            random_forest_predictions.ipynb
embedding_scaler.joblib      regression_models.ipynb
experiments.ipynb            tree_and_clinical_data_lstms.ipynb
final_model.ipynb            trees.txt
final_patient_ds.csv         tree_tumourid.csv
indices_cox_model.ipynb      visualization/
logistical_regression.ipynb


In [62]:
# -----------------------
# 3. Train/Validation split
# -----------------------
train_idx, val_idx = train_test_split(
    list(range(len(dataset))),
    test_size=0.2,
    random_state=random_seed
)
print(f"Train samples: {len(train_idx)}, Validation samples: {len(val_idx)}")

train_ds = torch.utils.data.Subset(dataset, train_idx)
val_ds = torch.utils.data.Subset(dataset, val_idx)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=16, shuffle=True, collate_fn=collate_trees_clinical
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=32, shuffle=False, collate_fn=collate_trees_clinical
)

Train samples: 169, Validation samples: 43


In [63]:
# -----------------------
# 4. Model
# -----------------------
clinical_dim = len(clinical_cols)
vocab_size = len(vocab)
print(f"Vocabulary size used for model instantiation: {vocab_size}")

model = TreeSurvivalModelWithClinical(
    vocab_size=vocab_size,     # size of your node ID vocabulary
    node_emb_dim=embed_dim,    # embedding size for node labels
    tree_hid=mem_dim,
    mlp_hidden=64,
    clinical_dim=len(clinical_cols)
).to(device)

if torch.isnan(model.emb.weight).any():
    print("!!! WARNING: NaN detected in initial embedding weights!")

print(f"Embedding layer weight size: {model.emb.weight.size()}")

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

Vocabulary size used for model instantiation: 58
Embedding layer weight size: torch.Size([58, 32])


In [64]:
print(f"Dataset length: {len(dataset)}")
print(f"Train set length: {len(train_ds)}")
print(f"Val set length: {len(val_ds)}")

Dataset length: 212
Train set length: 169
Val set length: 43


In [65]:
def train_epoch_treeclinical(model, optimizer, dataloader, device):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for batch_idx, (roots, clin_batch, times, events) in enumerate(dataloader):
        if torch.isnan(clin_batch).any():
            print(f"Batch {batch_idx}: NaN detected in clinical_batch BEFORE processing!")
            continue # Skip this batch
        optimizer.zero_grad()
        batch_risks = []

        for i, root in enumerate(roots):
            # Prepare clinical tensor
            clin = clin_batch[i] if clin_batch.numel() > 0 else torch.zeros((1, 0), device=device)
            if clin.dim() == 1:
                clin = clin.unsqueeze(0)
            clin = clin.to(device)

            # Forward pass
            risk = model(root, clin)
            if risk.dim() == 0:
                risk = risk.unsqueeze(0)
            batch_risks.append(risk)

        # Stack all risks into a single tensor
        risks = torch.stack(batch_risks).view(-1).to(device)
        times = times.to(device)
        events = events.to(device)

        # Add this assertion to check if risks require gradients
        assert risks.requires_grad, "Risks tensor does not require gradients!"

        # Debugging prints for NaN detection
        if torch.isnan(risks).any() or torch.isinf(risks).any():
             print(f"Batch {batch_idx}: NaN or Inf detected in risks:", risks)


        # Compute Cox loss
        loss = cox_ph_loss(risks, times, events)

        # Debugging prints for NaN detection in loss
        if math.isnan(loss.item()):
            print(f"Batch {batch_idx}: NaN detected in loss.")
            # Optionally print intermediate values from cox_ph_loss if needed

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(1, n_batches)

In [66]:
def eval_treeclinical(model, dataloader, device):
    model.eval()
    risks_all, times_all, events_all = [], [], []

    with torch.no_grad():
        for roots, clin_list, times, events in dataloader:
            for root, clin in zip(roots, clin_list):
                clin = clin.to(device) if clin.numel() > 0 else torch.tensor([], device=device)
                risk = model(root, clin)
                if risk.dim() == 0:
                    risk = risk.unsqueeze(0)
                risks_all.extend(risk.cpu().numpy())
            times_all.extend(times.numpy())
            events_all.extend(events.numpy())

    return concordance_index(np.array(risks_all), np.array(times_all), np.array(events_all))

In [67]:
import copy
from itertools import product

class EarlyStopping:
    """Stop training when validation C-index doesn't improve after patience epochs."""
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = -float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, score, model):
        improved = score > self.best_score + self.min_delta
        if improved:
            self.best_score = score
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


# -----------------------
# Hyperparameter grid
# -----------------------
param_grid = {
    "embed_dim": [16, 32],
    "mem_dim": [64, 128],
    "dropout": [0.1, 0.3],
    "lr": [1e-4, 5e-5, 1e-5], # Reduced learning rates
    "weight_decay": [1e-5, 1e-4],
}

In [68]:
dataset = TreeClinicalDatasetLines(
    tracerx_df, trees,
    cruk_col="cruk_id", line_col="line_index",
    time_col="os_time", event_col="event_os",
    clinical_cols=clinical_cols, vocab=vocab
)

print("Dataset length:", len(dataset))

train_idx, val_idx = train_test_split(
    list(range(len(dataset))),
    test_size=0.2,
    random_state=random_seed
)
print("Max train index:", max(train_idx))
print("Max val index:", max(val_idx))

train_ds = torch.utils.data.Subset(dataset, train_idx)
val_ds   = torch.utils.data.Subset(dataset, val_idx)
print(f"Dataset length: {len(dataset)}")
print(f"Train set length: {len(train_ds)}")
print(f"Val set length: {len(val_ds)}")

Dataset length: 212
Max train index: 211
Max val index: 210
Dataset length: 212
Train set length: 169
Val set length: 43


In [69]:
# Prepare DataLoaders
results = []

for embed_dim, mem_dim, dropout, lr, wd in product(
    param_grid["embed_dim"],
    param_grid["mem_dim"],
    param_grid["dropout"],
    param_grid["lr"],
    param_grid["weight_decay"],
):

    print(f"\n=== Training embed={embed_dim}, mem={mem_dim}, dropout={dropout}, lr={lr}, wd={wd} ===")

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_trees_clinical)
    val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_trees_clinical)

    model = TreeSurvivalModelWithClinical(
        vocab_size=len(vocab),
        node_emb_dim=embed_dim,
        tree_hid=mem_dim,
        mlp_hidden=64,
        clinical_dim=len(clinical_cols),
        dropout=dropout
    ).to(device)

    if torch.isnan(model.emb.weight).any():
      print("!!! WARNING: NaN detected in initial embedding weights!")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    early_stopper = EarlyStopping(patience=5)

    best_val = -1
    for epoch in range(1, 50):
        model.train()
        train_loss = train_epoch_treeclinical(model, optimizer, train_loader, device)

        # Add check for nan loss
        if math.isnan(train_loss):
            print(f"Epoch {epoch:02d} | Train loss is NaN. Stopping training for this config.")
            break

        val_cidx = eval_treeclinical(model, val_loader, device)
        print(f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | Val C-index: {val_cidx:.4f}")

        if early_stopper.step(val_cidx, model):
            print("Early stopping triggered.")
            break
        best_val = max(best_val, val_cidx)

    early_stopper.restore_best(model)
    results.append(((embed_dim, mem_dim, dropout, lr, wd), best_val))

# Print sorted results
results.sort(key=lambda x: -x[1])
print("\n=== Hyperparameter tuning results ===")
for params, val in results:
    print(f"embed={params[0]}, mem={params[1]}, dropout={params[2]}, lr={params[3]}, wd={params[4]} -> val_C={val:.4f}")


=== Training embed=16, mem=64, dropout=0.1, lr=0.0001, wd=1e-05 ===
Epoch 01 | Train loss: 1.4557 | Val C-index: 0.4163
Epoch 02 | Train loss: 1.4986 | Val C-index: 0.4475
Epoch 03 | Train loss: 1.4271 | Val C-index: 0.4747
Epoch 04 | Train loss: 1.4748 | Val C-index: 0.4825
Epoch 05 | Train loss: 1.4432 | Val C-index: 0.4942
Epoch 06 | Train loss: 1.4129 | Val C-index: 0.5214
Epoch 07 | Train loss: 1.4130 | Val C-index: 0.5175
Epoch 08 | Train loss: 1.4002 | Val C-index: 0.5136
Epoch 09 | Train loss: 1.4142 | Val C-index: 0.5175
Epoch 10 | Train loss: 1.4158 | Val C-index: 0.5253
Epoch 11 | Train loss: 1.4156 | Val C-index: 0.5370
Epoch 12 | Train loss: 1.4255 | Val C-index: 0.5603
Epoch 13 | Train loss: 1.3678 | Val C-index: 0.5720
Epoch 14 | Train loss: 1.3916 | Val C-index: 0.5603
Epoch 15 | Train loss: 1.3642 | Val C-index: 0.5720
Epoch 16 | Train loss: 1.3517 | Val C-index: 0.5720
Epoch 17 | Train loss: 1.3444 | Val C-index: 0.5798
Epoch 18 | Train loss: 1.3794 | Val C-index: 0.